# Bi-Encoder Retrieval vs. Cross-Encoder Re-Ranking — a RAG Case Study on the 2026 F1 Technical Regulations

This notebook builds and compares two retrieval-augmented generation (RAG) pipelines over **one** chunked document, the *FIA 2026 Formula 1 Regulations, Section C [Technical], Issue 20* (5 Aug 2026, 254 pages). The document is a dense legal and technical rulebook: numbered articles, cross-references, exceptions ("except for…", "notwithstanding…") and hundreds of prohibitions ("is forbidden", "may not").

| | Pipeline 1: Baseline | Pipeline 2: Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder (`all-MiniLM-L6-v2`) + FAISS → **top 10** | *same* top 10 from Pipeline 1 |
| Stage 2 | take the bi-encoder's top 3 | Cross-encoder (`ms-marco-MiniLM-L-6-v2`) re-scores all 10 → **top 3** |
| Generation | DeepSeek, shared prompt → **Answer 1** | DeepSeek, *identical* prompt → **Answer 2** |

**Contents**
1. Setup
2. Load, clean and chunk the PDF (done once, shared by both pipelines)
3. Build the vector index
4. The two pipelines
5. Failure-mode stress tests
6. Rank-shift analysis
7. Answer 1 vs. Answer 2
8. Latency profiling
9. Scaling: why not cross-encode everything?
10. Report

The reasoning behind each design choice is documented in `design_decisions.md`.

## 1. Setup

The models, the `k` values and the LLM settings are all constants here, so an experiment can be changed in one place.
The DeepSeek API key is read from a local `.env` file (`DEEPSEEK_API_KEY=sk-...`) and never appears in the notebook.

In [1]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # keep model-download/loading bars out of the outputs
import re
import unicodedata
import time
import random
import textwrap

import numpy as np
import pandas as pd
import pymupdf                      # PDF text extraction (PyMuPDF)
import faiss                        # vector store / exact similarity search
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI           # DeepSeek exposes an OpenAI-compatible API
from dotenv import load_dotenv
from IPython.display import display, Markdown
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()
hf_logging.set_verbosity_error()      # silence "sequence longer than max length" notices (measured explicitly in §3)

# ---------------- Experiment configuration ----------------
PDF_PATH        = "fia_2026_f1_technical_regulations.pdf"   # FIA 2026 F1 Regs, Section C [Technical], Iss 20
# Pages with regulatory prose. Skipped: p1-6 table of contents; p204-236 Appendices C2-C4 (CAD volume lists,
# drawings, PU component matrix); p244-254 Appendix C6 (classification tables). Tables extract as word salad.
INCLUDE_PAGES   = set(range(7, 204)) | set(range(237, 244))   # Articles C1-C18, App. C1 (Definitions), App. C5
CHUNK_WORDS     = 120              # target chunk size; technical text is ~1.4 tokens/word, so 120 words keeps
                                   # >99% of chunks inside the bi-encoder's 256-token window (checked below)
OVERLAP_WORDS   = 30               # overlap used only when one long clause has to be split

BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K = 10                         # candidates retrieved by the bi-encoder
TOP_N = 3                          # chunks passed to the LLM

LLM_BASE_URL    = "https://api.deepseek.com"
LLM_MODEL       = "deepseek-flash"  # DeepSeek's fast chat model ("deepseek-chat" is an alias of it)
LLM_TEMPERATURE = 0.0               # (near-)deterministic answers -> fair A/B comparison
LLM_MAX_TOKENS  = 400
LLM_THINKING    = "disabled"        # deepseek-flash is a reasoning model: hidden "thinking" would consume the
                                    # token budget (empty answers) and add latency; the task here is pure
                                    # reading comprehension over the supplied excerpts, so thinking is off.

# ---------------- Reproducibility ----------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(4)            # fixed thread count -> more stable latency numbers

pd.set_option("display.max_colwidth", 120)

# ---------------- API key (fail fast with a clear message) ----------------
load_dotenv()
assert os.getenv("DEEPSEEK_API_KEY"), "DEEPSEEK_API_KEY missing: create a .env file next to this notebook."
llm_client = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url=LLM_BASE_URL)
print("Setup OK | torch", torch.__version__, "| faiss", faiss.__version__)

Setup OK | torch 2.14.1 | faiss 1.15.1


## 2. Load, clean and chunk the PDF

Both pipelines must use **exactly the same chunked corpus**, so chunking happens once, here.

**Cleaning.** PyMuPDF extracts the text page by page. These artefacts specific to this PDF are removed or fixed:
- **Running header.** Every page starts with a 10-line block ("SECTION C: TECHNICAL REGULATIONS … Issue 20"), which would otherwise be repeated in every chunk.
- **Broken "ff" ligature.** The PDF's font maps the "ff" ligature to the letter `a`, so "effect" is extracted as *eaect* and "differ" as *diaer*. The ~20 affected word stems were found by comparing tokens against an English word list. They are repaired with an explicit map, which is transparent and portable.
- **Unicode cleanup.** `ﬁ`/`ﬂ` ligatures, Unicode minus signs and table tick marks (`ü`, `û`) are normalised or removed.
- **Governance boilerplate.** "Advisory Committee: …" and "Governance: …" lines repeat under every article and carry no rule content.

**Clause-aware chunking.** A regulation is a tree of numbered clauses (`C5.1` → `C5.1.3`). Chunks follow that structure rather than cutting at arbitrary word counts:
1. The text is split into **clause units** wherever a line holds only a clause number (`C5.1.3`, or `1.6` in Appendix C5). Each unit is tagged with its **section**, the 2-level clause and its title (e.g. `C9.7 Reverse gear`), and with its page.
2. Consecutive units of the same section are packed into chunks of up to about 120 words. A chunk never mixes two sections.
3. A single clause longer than 120 words is split at sentence boundaries, with a 30-word overlap.
4. If a chunk starts mid-section, its section title is prepended (e.g. `C5.20 ES design and installation: C5.20.8 …`), so the clause is never stripped of its context. Both encoders see exactly the same text.

In [2]:
# ---------- 2a. Cleaning ----------
HEADER_RE    = re.compile(r"\A.*?Issue 20\s*\n\s*0\s*\n\s*C\s*\n", re.S)   # running header block on every page
DROP_LINE_RE = re.compile(r"^(Advisory Committee:|Governance:)")           # governance boilerplate lines

# The PDF's font maps the "ff" ligature to the letter "a" ("effect" -> "eaect"). Stems found by checking
# extracted tokens against an English word list; repaired explicitly (word-start anchored).
FF_FIXES = {
    "eaect": "effect", "aaect": "affect", "diaer": "differ", "stianess": "stiffness", "stiaen": "stiffen",
    "paraain": "paraffin", "eaicien": "efficien", "suaicien": "sufficien", "aaix": "affix", "suaix": "suffix",
    "oaicial": "official", "buaer": "buffer", "eaort": "effort", "oaset": "offset", "midria": "midriff",
    "oaer": "offer", "oa": "off",
}
FF_RE = re.compile(r"\b(" + "|".join(sorted(FF_FIXES, key=len, reverse=True)) + r")", re.I)


def fix_ff(match: re.Match) -> str:
    word = match.group(0); fixed = FF_FIXES[word.lower()]
    return fixed.capitalize() if word[0].isupper() else fixed


def clean_page_text(text: str) -> str:
    """Remove the running header and repair character-level extraction artefacts."""
    text = HEADER_RE.sub("", text, count=1)
    text = unicodedata.normalize("NFKC", text)            # ﬁ -> fi, ﬂ -> fl, etc.
    text = text.replace("−", "-").replace("–", "-")       # unicode minus / en dash -> hyphen
    text = re.sub(r"[üû]", " ", text)                     # tick/cross symbols from tables (Wingdings font)
    return FF_RE.sub(fix_ff, text)


# ---------- 2b. Parse into clause units ----------
CLAUSE_RE         = re.compile(r"^(C\d+(?:\.\d+)+|\d+\.\d+(?:\.\d+)*)$")      # line holding only a clause number
SECTION_RE        = re.compile(r"^C\d+\.\d+$")                               # 2-level number -> next line = title
SECTION_INLINE_RE = re.compile(r"^(C\d+\.\d+) ([A-Z][^.]{2,80})$")           # "C17.3 Standard Supply Components"
HEADING_RE        = re.compile(r"^(ARTICLE C\d+:.*|APPENDIX C\d+:.*|PART [A-Z]:.*)$")

doc = pymupdf.open(PDF_PATH)
lines = []                                                 # (page number, cleaned line)
for page_no, page in enumerate(doc, start=1):
    if page_no not in INCLUDE_PAGES:
        continue
    for line in clean_page_text(page.get_text()).split("\n"):
        line = re.sub(r"\s+", " ", line).strip()
        if line and not DROP_LINE_RE.match(line):
            lines.append((page_no, line))

units, current, section = [], None, ""
for idx, (page_no, line) in enumerate(lines):
    if HEADING_RE.match(line):                             # ARTICLE / APPENDIX headings: metadata only
        section, current = line, None
        continue
    if SECTION_INLINE_RE.match(line) or CLAUSE_RE.match(line):
        if SECTION_INLINE_RE.match(line):
            section = line
        elif SECTION_RE.match(line) and idx + 1 < len(lines):
            section = f"{line} {lines[idx + 1][1]}"         # e.g. "C9.7" + "Reverse gear"
        current = {"page": page_no, "section": section, "lines": []}
        units.append(current)
    elif current is None:                                   # text right after a heading, before any number
        current = {"page": page_no, "section": section, "lines": []}
        units.append(current)
    current["lines"].append(line)


def unit_text(unit_lines: list[str]) -> str:
    text = " ".join(unit_lines)
    text = re.sub(r"(\w)- (?=[a-z])", r"\1-", text)         # re-join words hyphenated across lines
    return re.sub(r"\s+", " ", text).strip()


def split_sentences(text: str) -> list[str]:
    """Split before a capital letter or a list marker ("a.", "ii.") that follows . ; or :"""
    return re.split(r"(?<=[.;:])\s+(?=[A-Z(\"“]|[a-z]\.\s|[ivx]+\.\s)", text)


# ---------- 2c. Pack clause units into chunks ----------
chunks = []                                                 # the single shared corpus


def emit(section: str, page: int, text: str):
    if not text.startswith(section.split()[0]):            # chunk starts mid-section -> prepend section title
        text = f"{section}: {text}"
    chunks.append({"chunk_id": len(chunks), "page": page, "section": section, "text": text})


buffer, buf_section, buf_page = [], None, None


def flush():
    global buffer
    if buffer:
        emit(buf_section, buf_page, " ".join(buffer))
    buffer = []


for unit in units:
    text = unit_text(unit["lines"])
    n_words = len(text.split())
    if not text:
        continue
    # start a new chunk on a section change or when the size budget would be exceeded
    if unit["section"] != buf_section or sum(len(b.split()) for b in buffer) + n_words > CHUNK_WORDS:
        flush(); buf_section, buf_page = unit["section"], unit["page"]
    if n_words <= CHUNK_WORDS:
        buffer.append(text)
        continue
    # one clause longer than the budget: split at sentence boundaries with overlap
    flush()
    part = []
    for sentence in split_sentences(text):
        part.append(sentence)
        if sum(len(s.split()) for s in part) >= CHUNK_WORDS:
            emit(unit["section"], unit["page"], " ".join(part))
            tail, carried = [], 0
            for s in reversed(part):                        # carry ~OVERLAP_WORDS into the next piece
                if carried >= OVERLAP_WORDS:
                    break
                tail.insert(0, s); carried += len(s.split())
            part = tail
    if part:
        emit(unit["section"], unit["page"], " ".join(part))
flush()

chunk_texts = [c["text"] for c in chunks]
lengths = np.array([len(t.split()) for t in chunk_texts])
print(f"{len(INCLUDE_PAGES)} of {doc.page_count} pages -> {len(units)} clause units -> {len(chunks)} chunks | "
      f"words/chunk: mean {lengths.mean():.0f}, median {np.median(lengths):.0f}, max {lengths.max()}")
print(f"Distinct sections: {len({c['section'] for c in chunks})}")
for cid in (2, 480):
    print(f"\nSample chunk {cid} (p.{chunks[cid]['page']}):\n" + textwrap.fill(chunks[cid]["text"], 110))

204 of 254 pages -> 958 clause units -> 1032 chunks | words/chunk: mean 98, median 107, max 185
Distinct sections: 180

Sample chunk 2 (p.7):
C1.3 Compliance with the regulations Formula 1 Cars must comply with these regulations in their entirety at
all times during a Competition. Should a Competitor or PU Manufacturer introduce a new design or system or
feel that any aspect of these regulations is unclear, clarification may be sought from the FIA Formula One
Technical Department. If clarification relates to any new design or system, correspondence must include: a. A
full description of the design or system. b. Drawings or schematics where appropriate. c. The Competitor or PU
Manufacturer's opinion concerning the immediate implications on other parts of the car of any proposed new
design. d. The Competitor or PU Manufacturer's opinion concerning any possible long-term consequences or new
developments which may come from using any such new designs or systems.

Sample chunk 480 (p.104):


## 3. Build the vector index (bi-encoder + FAISS)

- Every chunk is embedded **once** with the bi-encoder.
- The vectors are L2-normalised, so the **inner product equals cosine similarity**.
- They are stored in a FAISS `IndexFlatIP`. This is an *exact* (brute-force) index, so any ranking error is the embedding model's fault, not an approximate search's.

Both models are loaded here, so that loading time never leaks into the latency measurements later.

In [3]:
bi_encoder    = SentenceTransformer(BI_ENCODER_NAME)
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)

t0 = time.perf_counter()
chunk_embeddings = bi_encoder.encode(chunk_texts, normalize_embeddings=True, batch_size=64,
                                     convert_to_numpy=True).astype("float32")
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])   # inner product on unit vectors == cosine
index.add(chunk_embeddings)
index_time = time.perf_counter() - t0

# How many chunks exceed each encoder's maximum input length (and would be silently truncated)?
n_tok = np.array([len(bi_encoder.tokenizer(t)["input_ids"]) for t in chunk_texts])
print(f"Tokens/chunk: mean {n_tok.mean():.0f}, p95 {np.percentile(n_tok, 95):.0f} | truncated by bi-encoder "
      f"(>{bi_encoder.max_seq_length}): {(n_tok > bi_encoder.max_seq_length).sum()} | by cross-encoder (>512): "
      f"{(n_tok > 512).sum()}")
print(f"Indexed {index.ntotal} chunks, dim={chunk_embeddings.shape[1]}, offline indexing time {index_time:.2f}s")

Tokens/chunk: mean 138, p95 214 | truncated by bi-encoder (>256): 4 | by cross-encoder (>512): 0
Indexed 1032 chunks, dim=384, offline indexing time 2.13s


## 4. The two pipelines

The building blocks are shared, so that **the only difference between the pipelines is the ranking step**:

| Function | Role |
|---|---|
| `retrieve(query)` | Stage 1. Embeds the query, searches FAISS and returns the top-10 candidates with `bi_score` and `bi_rank`. |
| `rerank(query, candidates)` | Stage 2. Scores **exactly those 10** (query, chunk) pairs with the cross-encoder and adds `ce_score` and `ce_rank`. It never touches the index. |
| `build_prompt(query, chunks)` | **One** prompt template used by both pipelines. |
| `generate(prompt)` | Calls DeepSeek. |

Every stage is timed with `time.perf_counter()`.

In [4]:
def retrieve(query: str, k: int = TOP_K):
    """Stage 1 (bi-encoder): return the top-k chunks by cosine similarity, plus timings."""
    t0 = time.perf_counter()
    q_emb = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    t1 = time.perf_counter()
    scores, ids = index.search(q_emb, k)                 # exact top-k inner-product search
    t2 = time.perf_counter()
    candidates = [{"chunk_id": int(cid), "page": chunks[cid]["page"], "bi_score": float(s), "bi_rank": r + 1}
                  for r, (cid, s) in enumerate(zip(ids[0], scores[0]))]
    return candidates, {"embed_query_ms": (t1 - t0) * 1e3, "faiss_search_ms": (t2 - t1) * 1e3}


def rerank(query: str, candidates: list[dict]):
    """Stage 2 (cross-encoder): jointly score each (query, chunk) pair and re-sort the SAME candidates."""
    t0 = time.perf_counter()
    pairs = [(query, chunks[c["chunk_id"]]["text"]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs, batch_size=len(pairs))   # all 10 pairs in one batch
    t1 = time.perf_counter()
    reranked = [dict(c, ce_score=float(s)) for c, s in zip(candidates, ce_scores)]
    reranked.sort(key=lambda c: c["ce_score"], reverse=True)          # descending cross-encoder score
    for r, c in enumerate(reranked):
        c["ce_rank"] = r + 1
    return reranked, {"cross_encode_ms": (t1 - t0) * 1e3}


PROMPT_TEMPLATE = """You are an assistant answering questions about the FIA 2026 Formula 1 Technical Regulations.
Answer the question using ONLY the regulation excerpts below. Do not use outside knowledge.
If the excerpts do not contain enough information to answer, say so explicitly.
Cite the chunk IDs and article numbers you relied on, e.g. [Chunk 12, C5.1.3]. Keep the answer under 120 words.

Regulation excerpts:
{context}

Question: {question}
Answer:"""


def build_prompt(query: str, selected: list[dict]) -> str:
    """The single prompt template shared by BOTH pipelines; only `selected` differs."""
    context = "\n\n".join(f"[Chunk {c['chunk_id']}, p.{c['page']}]\n{chunks[c['chunk_id']]['text']}"
                          for c in selected)
    return PROMPT_TEMPLATE.format(context=context, question=query)


def generate(prompt: str):
    """Call the LLM and return (answer, latency in ms)."""
    t0 = time.perf_counter()
    resp = llm_client.chat.completions.create(
        model=LLM_MODEL, temperature=LLM_TEMPERATURE, max_tokens=LLM_MAX_TOKENS,
        messages=[{"role": "user", "content": prompt}],
        extra_body={"thinking": {"type": LLM_THINKING}})       # DeepSeek-specific request field
    elapsed_ms = (time.perf_counter() - t0) * 1e3
    choice = resp.choices[0]
    assert choice.message.content and choice.finish_reason == "stop", \
        f"LLM returned an empty/truncated answer (finish_reason={choice.finish_reason})"
    return choice.message.content.strip(), elapsed_ms


def pipeline1(query: str, call_llm: bool = True):
    """Baseline: bi-encoder top-10 -> take top-3 directly -> LLM (Answer 1)."""
    candidates, timings = retrieve(query)
    selected = candidates[:TOP_N]
    answer, timings["llm_ms"] = generate(build_prompt(query, selected)) if call_llm else (None, np.nan)
    return {"candidates": candidates, "selected": selected, "answer": answer, "timings": timings}


def pipeline2(query: str, call_llm: bool = True, candidates: list[dict] | None = None):
    """Re-ranking: the SAME bi-encoder top-10 -> cross-encoder re-sort -> top-3 -> LLM (Answer 2).

    `candidates` can be passed in from pipeline1 so that both pipelines provably share one candidate set;
    if omitted, stage 1 is run here (identical, deterministic result) and timed as part of this pipeline.
    """
    timings = {}
    if candidates is None:
        candidates, timings = retrieve(query)
    reranked, ce_t = rerank(query, candidates)
    timings.update(ce_t)
    selected = reranked[:TOP_N]
    answer, timings["llm_ms"] = generate(build_prompt(query, selected)) if call_llm else (None, np.nan)
    return {"candidates": reranked, "selected": selected, "answer": answer, "timings": timings}

**Sanity checks.** These verify the spec's invariants before running any experiment:
- Pipeline 2 re-ranks Pipeline 1's exact candidates.
- Bi-encoder scores come out in descending order.
- Both pipelines share one prompt template.

In [5]:
_q = "What is the minimum mass of the car?"
_p1 = pipeline1(_q, call_llm=False)
_p2 = pipeline2(_q, call_llm=False, candidates=_p1["candidates"])

ids_p1 = [c["chunk_id"] for c in _p1["candidates"]]
assert len(ids_p1) == TOP_K
assert {c["chunk_id"] for c in _p2["candidates"]} == set(ids_p1), "Pipeline 2 must re-rank Pipeline 1's top-10"
assert {c["chunk_id"] for c in _p2["selected"]} <= set(ids_p1)
assert all(a["bi_score"] >= b["bi_score"] for a, b in zip(_p1["candidates"], _p1["candidates"][1:]))
assert all(a["ce_score"] >= b["ce_score"] for a, b in zip(_p2["candidates"], _p2["candidates"][1:]))
assert pipeline2(_q, call_llm=False)["candidates"] == _p2["candidates"], "Stage 1 must be deterministic"
print("All sanity checks passed.")

All sanity checks passed.


## 5. Failure-mode stress tests

**How the queries were chosen.**
- I extracted every sentence containing a prohibition or exception marker ("may not", "forbidden", "except for", "notwithstanding", …): about 300 of them.
- From those, plus well-known topics (mass, fuel, tyres, lights), I wrote about **70 questions the way a fan or a junior engineer would ask them**, deliberately *not* reusing the regulation's wording.
- Each one was run through both rankers. I kept the queries where the clause that actually answers the question (the **gold chunk**) sits **outside the bi-encoder's top 3** and the cross-encoder lifts it **into** the top 3.

Most of the pool did **not** fail. The section titles prepended to each chunk ("C9.7 Reverse gear") make the bi-encoder strong on topical questions. The survivors are the cases where the answer hides in an *exception, prohibition or side-clause* of a broader section.

**How gold chunks are identified.** Each gold chunk is pinned down by a distinctive *gold phrase* that appears only in it. The chunk ID is computed from the phrase, not hard-coded, so this keeps working if the chunking parameters change.

Two **control queries** are also included, where re-ranking does *not* fix the problem. They show the method's limits honestly.

In [6]:
FAILURE_QUERIES = [
    # (id, category, query, gold phrase that identifies the answering chunk)
    ("Q1", "Negation / prohibition",
     "Is a launch control that detects the start lights allowed?",
     "detect when a race start signal is given"),
    ("Q2", "Keyword trap / synonym",
     "Are rotary engines allowed?",
     "Only 4-stroke engines with reciprocating pistons are permitted"),
    ("Q3", "Exception / corner case",
     "Must every part of the car be shown legal by physically inspecting the hardware?",
     "except for safety features"),
    ("Q4", "Constraint / role limit",
     "Can the driver cancel a driver default during the race start?",
     "may not be turned off before the end of that period"),
    ("Q5", "Negation / anti-circumvention",
     "Is it allowed to build a system that cheats the wing flex tests?",
     "circumventing the tests defined in this Article"),
    ("Q6", "Buried chunk",
     "Can the wheel nut be used to hold other parts?",
     "wheel fastener may not attach or mount any part"),
    ("Q7", "Buried chunk / paraphrase",
     "How much can the battery charge level swing during a lap?",
     "may not exceed 4MJ"),
    # ---- controls: cases where re-ranking does NOT rescue the answer ----
    ("C1", "Control: recall failure",
     "Can engine power depend on the clutch?",
     "must not be influenced by clutch position"),
    ("C2", "Control: no change",
     "Can teams top up oil or water during the race?",
     "no substance may be added to the car during the Race"),
]


def gold_excerpt(cid: int, phrase: str, width: int = 140) -> str:
    """Text window around the gold phrase, with the phrase in bold."""
    text = chunks[cid]["text"]
    i = text.lower().find(phrase.lower())
    start, end = max(0, i - width), min(len(text), i + len(phrase) + width)
    return ("…" if start else "") + text[start:i] + "**" + text[i:i + len(phrase)] + "**" + \
           text[i + len(phrase):end] + ("…" if end < len(text) else "")


def gold_chunk_id(phrase: str) -> int:
    """Return the unique chunk containing `phrase` (case-insensitive)."""
    hits = [c["chunk_id"] for c in chunks if phrase.lower() in c["text"].lower()]
    assert len(hits) == 1, f"gold phrase {phrase!r} matched {hits}; make it more specific"
    return hits[0]


results = {}
for qid, category, query, phrase in FAILURE_QUERIES:
    gold = gold_chunk_id(phrase)
    p1 = pipeline1(query)                                     # Answer 1
    p2 = pipeline2(query, candidates=p1["candidates"])        # Answer 2 (same top-10 candidates)
    results[qid] = {"category": category, "query": query, "phrase": phrase, "gold": gold, "p1": p1, "p2": p2}
    print(f"{qid} done | gold chunk {gold}")

Q1 done | gold chunk 410


Q2 done | gold chunk 242


Q3 done | gold chunk 5


Q4 done | gold chunk 409


Q5 done | gold chunk 206


Q6 done | gold chunk 535


Q7 done | gold chunk 257


C1 done | gold chunk 318


C2 done | gold chunk 237


## 6. Rank-shift analysis

**Summary.**
- *Bi-rank* is the gold chunk's position in the bi-encoder's top 10.
- *CE-rank* is its position after cross-encoder re-ranking.
- "–" means the chunk was not retrieved at all, so the re-ranker never saw it.

In [7]:
def rank_of(cands, cid, key):
    return next((c[key] for c in cands if c["chunk_id"] == cid), None)

rows = []
for qid, r in results.items():
    bi_r = rank_of(r["p1"]["candidates"], r["gold"], "bi_rank")
    ce_r = rank_of(r["p2"]["candidates"], r["gold"], "ce_rank")
    rows.append({
        "Query": qid, "Category": r["category"], "Gold chunk": r["gold"],
        "Gold section": chunks[r["gold"]]["section"][:40],
        "Page": chunks[r["gold"]]["page"],
        "Bi-encoder rank": f"#{bi_r}" if bi_r else "–",
        "Cross-encoder rank": f"#{ce_r}" if ce_r else "–",
        "Shift": f"#{bi_r} → #{ce_r}" if bi_r else "not retrieved",
        "In LLM context? (P1 / P2)": f"{'yes' if bi_r and bi_r <= TOP_N else 'NO'} / "
                                     f"{'yes' if ce_r and ce_r <= TOP_N else 'NO'}",
    })
rank_summary = pd.DataFrame(rows).set_index("Query")
rank_summary

,Category,Gold chunk,Gold section,Page,Bi-encoder rank,Cross-encoder rank,Shift,In LLM context? (P1 / P2)
Query,,,,,,,,
Q1,Negation / prohibition,410,C8.3 Start systems,91,#4,#1,#4 → #1,NO / yes
Q2,Keyword trap / synonym,242,C5.1 Engine specification,61,#8,#1,#8 → #1,NO / yes
Q3,Exception / corner case,5,C1.5 Duty of Competitor and PU Manufactu,8,#6,#2,#6 → #2,NO / yes
Q4,Constraint / role limit,409,C8.2 Control electronics,91,#5,#1,#5 → #1,NO / yes
Q5,Negation / anti-circumvention,206,C3.18 Aerodynamic Component Flexibility,53,#6,#1,#6 → #1,NO / yes
Q6,Buried chunk,535,C10.9 Wheel attachment and retention,115,#7,#3,#7 → #3,NO / yes
Q7,Buried chunk / paraphrase,257,C5.2 Power Unit Energy Flow,64,#4,#3,#4 → #3,NO / yes
C1,Control: recall failure,318,C5.13 Power unit control,74,–,–,not retrieved,NO / NO
C2,Control: no change,237,C4.4 Adding during the Race or Sprint Se,60,#6,#6,#6 → #6,NO / NO


**Detailed view per query.** The table shows all 10 candidates in bi-encoder order. ★ marks the gold chunk, and a 🔼 or 🔽 arrow shows how far the cross-encoder moved each chunk.

In [8]:
def candidate_table(qid: str) -> pd.DataFrame:
    r = results[qid]
    ce_by_id = {c["chunk_id"]: c for c in r["p2"]["candidates"]}
    rows = []
    for c in r["p1"]["candidates"]:
        ce = ce_by_id[c["chunk_id"]]
        delta = c["bi_rank"] - ce["ce_rank"]
        rows.append({
            "Gold": "★" if c["chunk_id"] == r["gold"] else "",
            "Chunk": c["chunk_id"], "Section": chunks[c["chunk_id"]]["section"][:32], "Page": c["page"],
            "Bi rank": c["bi_rank"], "Bi score (cos)": round(c["bi_score"], 3),
            "CE rank": ce["ce_rank"], "CE score (logit)": round(ce["ce_score"], 2),
            "Move": (f"🔼{delta}" if delta > 0 else f"🔽{-delta}" if delta < 0 else "·"),
            "Text (start)": chunks[c["chunk_id"]]["text"][:90] + "…",
        })
    return pd.DataFrame(rows).set_index("Bi rank")

for qid, r in results.items():
    display(Markdown(f"#### {qid} · {r['category']}  \n*{r['query']}*  \n"
                     f"Gold chunk {r['gold']} ({chunks[r['gold']]['section']}, p.{chunks[r['gold']]['page']}): "
                     f"“{gold_excerpt(r['gold'], r['phrase'])}”"))
    display(candidate_table(qid))

#### Q1 · Negation / prohibition  
*Is a launch control that detects the start lights allowed?*  
Gold chunk 410 (C8.3 Start systems, p.91): “C8.3 Start systems C8.3.1 Any system, the purpose and/or effect of which is to **detect when a race start signal is given**, is not permitted. C8.3.2 The FIA Standard ECU will implement a “lockout” period after each race start or pit stop during which a number of…”

,Gold,Chunk,Section,Page,Bi score (cos),CE rank,CE score (logit),Move,Text (start)
Bi rank,,,,,,,,,
1,,434,C8.12 Track signal information d,96,0.397,5,-9.91,🔽4,"C8.12 Track signal information display All cars must be fitted with red, blue and yellow c…"
2,,336,C5.17 Energy Recovery System (ER,77,0.393,3,-8.63,🔽1,C5.17.3 All cars must be fitted with two ERS status lights which: a. Have been supplied by…
3,,715,C14.3 Rear lights,149,0.389,6,-10.23,🔽3,C14.3 Rear lights C14.3.1 All cars must have three rear lights which: a. Have been supplie…
4,★,410,C8.3 Start systems,91,0.388,1,0.23,🔼3,"C8.3 Start systems C8.3.1 Any system, the purpose and/or effect of which is to detect when…"
5,,731,C14.7 Lateral Safety lights,152,0.379,8,-10.66,🔽3,C14.7 Lateral Safety lights C14.7.1 All cars must be fitted with two lateral safety lights…
6,,408,C8.2 Control electronics,91,0.370,9,-10.82,🔽3,C8.2.3 The control system wiring loom connectivity must be approved by the FIA. All wiring…
7,,435,C8.13 Impact warning system,97,0.367,2,-8.26,🔼5,C8.13 Impact warning system In order to give rescue crews an immediate indication of accid…
8,,422,C8.7 Master switch,94,0.355,4,-9.69,🔼4,"C8.7 Master switch C8.7.1 The driver, when seated normally with the safety belts fastened …"
9,,709,C14.1 Fire Extinguishers,148,0.333,10,-11.10,🔽1,C14.1 Fire Extinguishers: The triggering system must have its own source of energy and it …


#### Q2 · Keyword trap / synonym  
*Are rotary engines allowed?*  
Gold chunk 242 (C5.1 Engine specification, p.61): “C5.1 Engine specification C5.1.1 **Only 4-stroke engines with reciprocating pistons are permitted**. C5.1.2 Engine cubic capacity must be 1600cc (+0/-10cc). C5.1.3 All engines must have six cylinders arranged in a 90° “V” configuration and…”

,Gold,Chunk,Section,Page,Bi score (cos),CE rank,CE score (logit),Move,Text (start)
Bi rank,,,,,,,,,
1,,345,C5.18 MGU-K,79,0.472,3,-4.84,🔽2,C5.18.8 All rotating parts of the MGU-K and its mechanical transmission must have a fixed …
2,,328,C5.14 Engine high rev limits,76,0.447,4,-7.16,🔽2,"C5.14 Engine high rev limits: Except for the above conditions, Power Unit actuators may no…"
3,,767,C15.7 Materials and construction,161,0.433,7,-8.67,🔽4,C15.7.7 Reciprocating and rotating components: a. Reciprocating and rotating components mu…
4,,765,C15.7 Materials and construction,160,0.424,5,-7.53,🔽1,C15.7.4 Crankshafts must be manufactured from an iron-based alloy. No welding is permitted…
5,,243,C5.1 Engine specification,61,0.415,2,-0.46,🔼3,C5.1.4 Engines must have two intake and two exhaust valves per cylinder. Only reciprocatin…
6,,785,C16.1 Basic principles,164,0.410,9,-9.54,🔽3,"C16.1.6 Any fuel or engine oil, which appears to have been formulated in order to subvert …"
7,,327,C5.14 Engine high rev limits,76,0.401,6,-8.44,🔼1,C5.14 Engine high rev limits Engine high rev limits may vary for differing conditions prov…
8,★,242,C5.1 Engine specification,61,0.395,1,-0.01,🔼7,C5.1 Engine specification C5.1.1 Only 4-stroke engines with reciprocating pistons are perm…
9,,975,PART B: DEFINITIONS APPROVED UND,195,0.390,10,-9.82,🔽1,PART B: DEFINITIONS APPROVED UNDER THE PROVISIONS OF THE PU GOVERNANCE AGREEMENT: If the M…


#### Q3 · Exception / corner case  
*Must every part of the car be shown legal by physically inspecting the hardware?*  
Gold chunk 5 (C1.5 Duty of Competitor and PU Manufacturer, p.8): “…a Formula 1 Car, this duty and responsibility also extends to the PU Manufacturer. The design of the car, its components and systems shall, **except for safety features**, demonstrate their compliance with these regulations by means of physical inspection of hardware or materials. Unless explicitly requested …”

...

,Gold,Chunk,Section,Page,Bi score (cos),CE rank,CE score (logit),Move,Text (start)
Bi rank,,,,,,,,,
1,,258,C5.2 Power Unit Energy Flow,64,0.509,2,-4.18,🔽1,"C5.2.10 Recharge, as measured at the CU-K HV DC Bus, must not exceed a limit of 8.5MJ in e…"
2,,259,C5.2 Power Unit Energy Flow,64,0.495,5,-7.02,🔽3,C5.2 Power Unit Energy Flow: This limit may be further reduced to no less than 5 4MJ for S…
3,,267,C5.2 Power Unit Energy Flow,66,0.412,1,-3.74,🔼2,"C5.2.18 With the exception of the ERS, the cumulative amount of energy stored in the Power…"
4,★,257,C5.2 Power Unit Energy Flow,64,0.387,3,-5.90,🔼1,C5.2.9 The difference between the maximum and the minimum state of charge of the ES may no…
5,,314,C5.12 Power unit torque or power,74,0.386,8,-10.83,🔽3,"C5.12.6 Subject to Article B7.2.1, the driver maximum power demand must not be reduced at …"
6,,313,C5.12 Power unit torque or power,74,0.362,6,-9.21,·,"C5.12.4 Except for conforming to Article C5.2.8, the driver maximum power demand cannot be…"
7,,251,C5.2 Power Unit Energy Flow,62,0.357,4,-6.37,🔼3,"C5.2 Power Unit Energy Flow C5.2.1 The use of any device, other than the engine described …"
8,,254,C5.2 Power Unit Energy Flow,64,0.355,9,-11.00,🔽1,"C5.2.8 Additionally, and subject to Article B7.2, the electrical DC power of the ERS-K use…"
9,,256,C5.2 Power Unit Energy Flow,64,0.331,7,-10.69,🔼2,C5.2 Power Unit Energy Flow: P(kW) = 250 when the car speed is below 310kph P(kW) = 1800 -…


#### C1 · Control: recall failure  
*Can engine power depend on the clutch?*  
Gold chunk 318 (C5.13 Power unit control, p.74): “…Teams may be required to demonstrate the accuracy of the Power Unit configurations used by the FIA Standard ECU. C5.13.3 Power unit control **must not be influenced by clutch position**, movement or operation. C5.13.4 The idle speed control target may not exceed 4,000rpm.”

,Gold,Chunk,Section,Page,Bi score (cos),CE rank,CE score (logit),Move,Text (start)
Bi rank,,,,,,,,,
1,,460,C9.2 Clutch control,101,0.482,3,-3.99,🔽2,C9.2.5 The amount by which the Clutch is engaged must be controlled solely and directly by…
2,,461,C9.2 Clutch control,101,0.426,5,-4.90,🔽3,"C9.2 Clutch control: When commanded by the driver, the amount of clutch engagement will be…"
3,,250,C5.1 Engine specification,62,0.406,9,-7.34,🔽6,"C5.1.17 The Clutch assembly, including but not limited to friction plates, springs, cover …"
4,,330,C5.16 Stall prevention systems,76,0.404,6,-5.47,🔽2,"C5.16 Stall prevention systems If a car is equipped with a stall prevention system, and in…"
5,,331,C5.16 Stall prevention systems,76,0.404,10,-7.38,🔽5,C5.16 Stall prevention systems: Each time such a system is activated the Clutch must be fu…
6,,453,C9.2 Clutch control,100,0.395,7,-6.48,🔽1,C9.2 Clutch control…
7,,327,C5.14 Engine high rev limits,76,0.380,1,-0.87,🔼6,C5.14 Engine high rev limits Engine high rev limits may vary for differing conditions prov…
8,,462,C9.2 Clutch control,101,0.371,8,-7.21,·,C9.2.6 When the clutch operating device is released from its maximum travel position it mu…
9,,480,C9.8 Gear changing,104,0.370,2,-3.66,🔼7,C9.8 Gear changing C9.8.1 Automatic gear changes are considered a driver aid and are there…


#### C2 · Control: no change  
*Can teams top up oil or water during the race?*  
Gold chunk 237 (C4.4 Adding during the Race or Sprint Session, p.60): “C4.4 Adding during the Race or Sprint Session With the exception of compressed gases, **no substance may be added to the car during the Race** or Sprint Session. If it becomes necessary to replace any part of the car during the Race or Sprint Session, the mass of the new part must …”

,Gold,Chunk,Section,Page,Bi score (cos),CE rank,CE score (logit),Move,Text (start)
Bi rank,,,,,,,,,
1,,813,C16.10 Sampling and testing at a,170,0.417,1,-0.49,·,"C16.10 Sampling and testing at a Competition C16.10.1 Each F1 Team must declare, prior to …"
2,,785,C16.1 Basic principles,164,0.399,7,-6.16,🔽5,"C16.1.6 Any fuel or engine oil, which appears to have been formulated in order to subvert …"
3,,388,C6.5 Fuel draining and sampling,87,0.393,4,-3.92,🔽1,C6.5 Fuel draining and sampling C6.5.1 F1 Teams must provide a means of removing all fuel …
4,,812,C16.9 Engine oil approval,170,0.387,9,-9.84,🔽5,C16.9 Engine oil approval C16.9.1 Before any engine oil may be used in the Formula One Cha…
5,,814,C16.10 Sampling and testing at a,171,0.367,2,-2.88,🔼3,C16.10.3 Engine oil samples taken during a Competition will be checked for conformity by u…
6,★,237,C4.4 Adding during the Race or S,60,0.364,6,-5.20,·,"C4.4 Adding during the Race or Sprint Session With the exception of compressed gases, no s…"
7,,1023,APPENDIX C5: HOMOLOGATION AND DE,241,0.364,5,-4.50,🔼2,"APPENDIX C5: HOMOLOGATION AND DEVELOPMENT OF POWER UNITS, FUEL AND OIL FOR: 4.1 During the…"
8,,815,C16.10 Sampling and testing at a,171,0.357,3,-3.78,🔼5,C16.10 Sampling and testing at a Competition: Samples which differ from the reference engi…
9,,1019,APPENDIX C5: HOMOLOGATION AND DE,240,0.350,8,-8.42,🔼1,"APPENDIX C5: HOMOLOGATION AND DEVELOPMENT OF POWER UNITS, FUEL AND OIL FOR: The first time…"


## 7. Answer 1 vs. Answer 2

Both answers were produced with the **identical prompt template**. The only difference is which three chunks went into the context.

In [9]:
for qid, r in results.items():
    ctx1 = [c["chunk_id"] for c in r["p1"]["selected"]]
    ctx2 = [c["chunk_id"] for c in r["p2"]["selected"]]
    gold = r["gold"]
    mark = lambda ids: ", ".join(f"**{i}★**" if i == gold else str(i) for i in ids)
    display(Markdown(
        f"---\n### {qid} · {r['category']}\n**Question:** {r['query']}\n\n"
        f"**Answer 1 — baseline** (context chunks {mark(ctx1)}):\n\n> " + r["p1"]["answer"].replace("\n", "\n> ") +
        f"\n\n**Answer 2 — re-ranked** (context chunks {mark(ctx2)}):\n\n> " + r["p2"]["answer"].replace("\n", "\n> ")))

---
### Q1 · Negation / prohibition
**Question:** Is a launch control that detects the start lights allowed?

**Answer 1 — baseline** (context chunks 434, 336, 715):

> The provided excerpts do not contain enough information to answer whether a launch control that detects the start lights is allowed. The excerpts cover track signal information display [Chunk 434, C8.12], ERS status lights [Chunk 336, C5.17.3], and rear lights [Chunk 715, C14.3], but none address launch control systems or detecting start lights.

**Answer 2 — re-ranked** (context chunks **410★**, 435, 336):

> No. C8.3.1 explicitly prohibits any system whose purpose and/or effect is to detect when a race start signal is given [Chunk 410, C8.3.1]. Additionally, the FIA Standard ECU implements a lockout period after each race start or pit stop freezing/disabling certain Power Unit and transmission functions [Chunk 410, C8.3.2].

---
### Q2 · Keyword trap / synonym
**Question:** Are rotary engines allowed?

**Answer 1 — baseline** (context chunks 345, 328, 767):

> The excerpts do not contain enough information to answer whether rotary engines are allowed. None of the provided chunks address engine architecture or type restrictions. [Chunk 345, C5.18.8–C5.18.9; Chunk 328, C5.14; Chunk 767, C15.7.7]

**Answer 2 — re-ranked** (context chunks **242★**, 243, 345):

> No. Only 4-stroke engines with reciprocating pistons are permitted [Chunk 242, C5.1.1]. Rotary engines are therefore not allowed.

---
### Q3 · Exception / corner case
**Question:** Must every part of the car be shown legal by physically inspecting the hardware?

**Answer 1 — baseline** (context chunks 6, 39, 37):

> No. The excerpts do not require every part to be shown legal by physical hardware inspection. Compliance of electronic systems may be assessed by inspection of hardware, software, and data [Chunk 6, C1.5]. Competitors may be required to submit calculations or run with analytic devices specified by the FIA [Chunk 6, C1.5]. Cars may be measured during a Competition to check conformance to CAD models and/or Reference Volumes [Chunk 39, C3.3.2]. CAD models may be requested by the FIA [Chunk 6, C1.5].

**Answer 2 — re-ranked** (context chunks 402, **5★**, 7):

> No. The design of the car, its components and systems shall, except for safety features, demonstrate compliance by means of physical inspection of hardware or materials, and no mechanical design may rely on software inspection unless an Article explicitly requests it [Chunk 5, C1.5]. However, this does not say every part must be shown legal by physical hardware inspection; the excerpt also allows calculations or FIA-specified analytic devices to further demonstrate compliance with C3.2.2 and C3.18.1 [Chunk 5, C1.5].

---
### Q4 · Constraint / role limit
**Question:** Can the driver cancel a driver default during the race start?

**Answer 1 — baseline** (context chunks 479, 439, 410):

> The excerpts do not contain enough information to answer whether a driver can cancel a driver default during the race start. [Chunk 410, C8.3] covers start systems and lockout periods but does not mention "driver default" cancellation.

**Answer 2 — re-ranked** (context chunks **409★**, 313, 410):

> No. Any driver default turned on during the start lockout period may not be turned off before the end of that period [Chunk 409, C8.2.4].

---
### Q5 · Negation / anti-circumvention
**Question:** Is it allowed to build a system that cheats the wing flex tests?

**Answer 1 — baseline** (context chunks 25, 680, 438):

> The excerpts do not contain enough information to answer whether a system that "cheats" the wing flex tests is allowed. [Chunk 25, C3.2.2] states that where an Aerodynamic Component complies with a test defined in Article C3.18, it will be considered compliant with the rigidity requirement for the deflection tested, but this does not address intentional circumvention of the tests. No excerpt explicitly prohibits or permits "cheating" the tests.

**Answer 2 — re-ranked** (context chunks **206★**, 208, 681):

> No. [Chunk 206, C3.18.1] states that any system, mechanism, or procedure that achieves, seeks to, or is suspected of circumventing the load/deflection tests in Article C3.18 is prohibited. It also requires bodywork subjected to those tests to be of rigid construction with nominally linear elastic behaviour up to the maximum load applied whilst running on track, unless stated otherwise.

---
### Q6 · Buried chunk
**Question:** Can the wheel nut be used to hold other parts?

**Answer 1 — baseline** (context chunks 156, 537, 158):

> The excerpts do not contain enough information to answer whether the wheel nut can be used to hold other parts. The provided text covers wheel component compliance [Chunk 158, C3.14] and dual stage device test requirements [Chunk 537, C10.9.4–C10.9.5], but neither addresses using the wheel nut to retain other components.

**Answer 2 — re-ranked** (context chunks 537, 705, **535★**):

> No. Under [Chunk 535, C10.9.1], the wheel fastener may not attach or mount any part to the car except the wheel assembly described in Article C10.7.

---
### Q7 · Buried chunk / paraphrase
**Question:** How much can the battery charge level swing during a lap?

**Answer 1 — baseline** (context chunks 258, 259, 267):

> The excerpts do not specify a permitted battery charge-level swing per lap. They give Recharge limits (8.5 MJ/lap, reducible to 7 MJ, and further to no less than 5–4 MJ for Sprint Qualifying/Qualifying, plus up to 0.5 MJ additional) [Chunk 258, C5.2.10; Chunk 259, C5.2], and a non-ERS stored-energy limit of 300 kJ with no more than 20 kJ recovered per lap above 2 kW [Chunk 267, C5.2.18]. None states a charge-level swing figure.

**Answer 2 — re-ranked** (context chunks 267, 258, **257★**):

> The excerpts state that the difference between the maximum and minimum state of charge of the ES may not exceed 4MJ at any time the car is on track [Chunk 257, C5.2.9]. Separately, recharge as measured at the CU-K HV DC Bus must not exceed 8.5MJ per lap, with possible reductions [Chunk 258, C5.2.10]. The specific "swing during a lap" is therefore governed by the 4MJ state-of-charge difference limit [Chunk 257, C5.2.9].

---
### C1 · Control: recall failure
**Question:** Can engine power depend on the clutch?

**Answer 1 — baseline** (context chunks 460, 461, 250):

> The excerpts do not contain enough information to answer whether engine power can depend on the clutch. The provided chunks cover clutch control (C9.2, C9.2.5) and clutch/flywheel classification and numerical values (C5.1.17, C5.1.18), but none address a relationship between engine power and the clutch. [Chunk 460, C9.2.5; Chunk 461, C9.2; Chunk 250, C5.1.17–C5.1.18]

**Answer 2 — re-ranked** (context chunks 327, 480, 460):

> Yes. C5.14 allows a lower engine rev limit when the driver Clutch request exceeds 95% of total available travel, used only to protect the engine following driver error [Chunk 327, C5.14(c)]. Also, C9.2.5 states clutch engagement commanded by the driver is expressed in the FIA Standard ECU as torque at the rear axle by applying a gain of 5200Nm/90% to the clutch paddle position between 5% and 95% [Chunk 460, C9.2.5].

---
### C2 · Control: no change
**Question:** Can teams top up oil or water during the race?

**Answer 1 — baseline** (context chunks 813, 785, 388):

> The provided excerpts do not contain enough information to answer whether teams can top up oil or water during the race. The excerpts cover oil declaration and sampling before a Competition [Chunk 813, C16.10.1–C16.10.2], oil formulation and lubrication restrictions [Chunk 785, C16.1.6–C16.1.8], and fuel draining and sampling [Chunk 388, C6.5.1–C6.5.2], but none address topping up oil or water during a race.

**Answer 2 — re-ranked** (context chunks 813, 814, 815):

> The excerpts do not contain enough information to answer whether teams can top up oil or water during the race. They only cover oil declaration, sampling, and conformity testing [Chunk 813, C16.10.1–C16.10.2; Chunk 814, C16.10.3; Chunk 815, C16.10].

## 8. Latency profiling

**Method.**
- After a warm-up run, each retrieval path runs **20 times per query** over all stress-test queries.
- Pipeline 1 retrieval = query embedding + FAISS search.
- Pipeline 2 retrieval = the same, plus cross-encoder scoring of the 10 pairs.
- LLM latency comes from the real generation calls in §5 (one per query per pipeline). It is reported **separately** because it is a network round-trip, and including it would bury the millisecond differences between the retrieval stages.

Everything runs on CPU with 4 threads. The numbers below are for this machine.

In [10]:
N_REPEATS = 20
queries = [r["query"] for r in results.values()]

for q in queries:                                   # warm-up (first calls pay one-off allocation costs)
    pipeline2(q, call_llm=False)

stage_rows = []
for q in queries:
    for _ in range(N_REPEATS):
        p1 = pipeline1(q, call_llm=False)
        p2 = pipeline2(q, call_llm=False)            # runs its own stage 1, so it's timed end-to-end
        t1, t2 = p1["timings"], p2["timings"]
        stage_rows.append({
            "P1: embed query": t1["embed_query_ms"],
            "P1: FAISS search": t1["faiss_search_ms"],
            "P1: retrieval total": t1["embed_query_ms"] + t1["faiss_search_ms"],
            "P2: cross-encode 10 pairs": t2["cross_encode_ms"],
            "P2: retrieval + re-rank total": t2["embed_query_ms"] + t2["faiss_search_ms"] + t2["cross_encode_ms"],
        })
lat = pd.DataFrame(stage_rows)

llm1 = np.array([r["p1"]["timings"]["llm_ms"] for r in results.values()])
llm2 = np.array([r["p2"]["timings"]["llm_ms"] for r in results.values()])

latency_table = pd.DataFrame({
    "mean (ms)":   lat.mean(),
    "std (ms)":    lat.std(),
    "median (ms)": lat.median(),
    "p95 (ms)":    lat.quantile(0.95),
}).round(2)
latency_table.loc[f"LLM call (P1, n={len(llm1)})"] = [llm1.mean(), llm1.std(), np.median(llm1), np.quantile(llm1, .95)]
latency_table.loc[f"LLM call (P2, n={len(llm2)})"] = [llm2.mean(), llm2.std(), np.median(llm2), np.quantile(llm2, .95)]
latency_table = latency_table.round(1)
display(latency_table)

p1_ret = lat["P1: retrieval total"].mean()
p2_ret = lat["P2: retrieval + re-rank total"].mean()
llm_mean = np.concatenate([llm1, llm2]).mean()
print(f"Re-ranking overhead: +{p2_ret - p1_ret:.1f} ms per query ({p2_ret / p1_ret:.1f}x the retrieval cost)")
print(f"End-to-end: P1 ≈ {p1_ret + llm_mean:.0f} ms, P2 ≈ {p2_ret + llm_mean:.0f} ms "
      f"-> re-ranking adds {100 * (p2_ret - p1_ret) / (p1_ret + llm_mean):.1f}% to the total")

,mean (ms),std (ms),median (ms),p95 (ms)
P1: embed query,6.6,0.5,6.5,7.3
P1: FAISS search,0.1,0.0,0.1,0.1
P1: retrieval total,6.6,0.5,6.6,7.4
P2: cross-encode 10 pairs,31.6,2.8,31.7,34.6
P2: retrieval + re-rank total,37.7,2.9,37.6,41.1
"LLM call (P1, n=9)",1378.7,180.3,1424.3,1610.8
"LLM call (P2, n=9)",1183.6,168.9,1150.0,1458.9


Re-ranking overhead: +31.0 ms per query (5.7x the retrieval cost)
End-to-end: P1 ≈ 1288 ms, P2 ≈ 1319 ms -> re-ranking adds 2.4% to the total


## 9. Scaling: why not cross-encode the whole collection?

- **Bi-encoder.** The documents are encoded **offline**, once. At query time the cost is one query encoding plus a vector search, which approximate indexes such as HNSW make sub-linear in the corpus size N.
- **Cross-encoder.** There is nothing to pre-compute, because each score needs the query and the chunk *together*. Every query therefore costs **N full transformer forward passes**.

The cell below runs the cross-encoder over **every** chunk of this regulation for one query, measuring the real cost, and then extrapolates to larger collections.

In [11]:
q = results["Q3"]["query"]
all_pairs = [(q, t) for t in chunk_texts]

ce_full = []
for _ in range(3):
    t0 = time.perf_counter(); cross_encoder.predict(all_pairs, batch_size=32); ce_full.append(time.perf_counter() - t0)
ce_full_ms = np.median(ce_full) * 1e3
per_pair_ms = ce_full_ms / len(all_pairs)

bi_query_ms = lat["P1: retrieval total"].median()
print(f"Cross-encoding ALL {len(all_pairs)} chunks: {ce_full_ms:.0f} ms  ({per_pair_ms:.2f} ms/pair)  "
      f"vs. bi-encoder retrieval {bi_query_ms:.1f} ms")

scale = pd.DataFrame({"Corpus size (chunks)": [len(chunks), 10_000, 100_000, 10_000_000]})
def human_time(sec: float) -> str:
    return f"{sec:,.1f} s" if sec < 120 else f"{sec / 60:,.1f} min" if sec < 7200 else f"{sec / 3600:,.1f} h"

scale["Cross-encoder over everything (per query)"] = (scale.iloc[:, 0] * per_pair_ms / 1e3).map(human_time)
scale["Two-stage (retrieve 10 + re-rank 10)"] = f"≈ {lat['P2: retrieval + re-rank total'].median():.0f} ms + ANN search"
scale

Cross-encoding ALL 1032 chunks: 2425 ms  (2.35 ms/pair)  vs. bi-encoder retrieval 6.6 ms


,Corpus size (chunks),Cross-encoder over everything (per query),Two-stage (retrieve 10 + re-rank 10)
0,1032,2.4 s,≈ 38 ms + ANN search
1,10000,23.5 s,≈ 38 ms + ANN search
2,100000,3.9 min,≈ 38 ms + ANN search
3,10000000,6.5 h,≈ 38 ms + ANN search


## 10. Report

*The numbers quoted below come from the run whose outputs are saved in this notebook. Exact timings vary by machine.*

### 10.1 Architecture comparison

| | Bi-encoder (`all-MiniLM-L6-v2`) | Cross-encoder (`ms-marco-MiniLM-L-6-v2`) |
|---|---|---|
| Input | The query and each chunk are encoded **separately** | `[CLS] query [SEP] chunk [SEP]` is encoded **together** |
| Output | One 384-d vector per text; relevance = cosine(q, d) | One relevance logit per (query, chunk) pair |
| Interaction | None until the final dot product. Each text is squeezed into a single vector *before* it sees the other | **Full self-attention across both texts in every layer**: each query token attends to each chunk token |
| Pre-computation | Document vectors are computed once, offline, and indexed | Impossible: the score depends on the pair |
| Strength | Very fast and scalable, with good topical recall | Precise relevance; sensitive to negation, roles and exact conditions |
| Weakness | Matches on *topic* and word overlap. Negation and qualifiers ("not", "only", "except for") barely move a pooled vector | Cost is linear in the number of candidates; can only re-rank what it is given |

The bi-encoder answers "is this chunk *about* the same thing?". The cross-encoder answers "does this chunk *answer this* question?".

### 10.2 Qualitative & rank-shift analysis

**Rank shifts.** On all seven failure queries, the gold clause sat *outside* the baseline's context window, at bi-rank #4 to #8. The cross-encoder promoted it into the top 3 every time, and to **#1 in five of seven**:

| Query | Failure type | Gold clause | Shift | Baseline top 3 instead contained… |
|---|---|---|---|---|
| Q1 launch control detecting start lights | Negation / prohibition | C8.3.1 (p.91) | **#4 → #1** | three *lights* clauses: track signal display, ERS status lights, rear lights |
| Q2 rotary engines | Keyword trap / synonym | C5.1.1 (p.61) | **#8 → #1** | *rotating* parts: MGU-K rotating parts, rev limits, materials for "reciprocating and rotating components" |
| Q3 must every part be inspected physically? | Exception / corner case | C1.5 (p.8) | **#6 → #2** | other C1.5 / C3.3 legality-checking chunks (software inspection, CAD measurement) |
| Q4 cancel a driver default at the start | Constraint | C8.2.4 (p.91) | **#5 → #1** | C9.7 *Reverse gear* ("…by the **driver** at any time…"), sensor installations, start systems |
| Q5 system that cheats the wing flex tests | Negation / anti-circumvention | C3.18.1 (p.53) | **#6 → #1** | C3.2.2 rigidity rule, front-impact *test* procedures, sensor installation |
| Q6 wheel nut holding other parts | Buried chunk | C10.9.1 (p.115) | **#7 → #3** | C3.14 *Wheel Components* bodywork geometry, dual-stage fastener tests |
| Q7 battery charge swing per lap | Buried chunk / paraphrase | C5.2.9 (p.64) | **#4 → #3** | neighbouring C5.2 energy limits (recharge 8.5 MJ/lap, non-ERS storage) |
| C1 engine power vs. clutch | Control: recall failure | C5.13.3 (p.74) | not in top 10 | clutch-control clauses (C9.2) |
| C2 topping up oil or water in the race | Control: no change | C4.4 (p.60) | #6 → #6 | oil sampling / approval clauses (C16) |

**Why the bi-encoder missed them.** In this document the bi-encoder is strong on *topical* questions, helped by the section titles prepended to each chunk. For about 51 of the ~67 probe questions the gold clause was already in its top 3. Of the 16 misses, re-ranking rescued 7 (Q1–Q7). In 7 the gold clause was not in the top 10 at all, a recall failure no re-ranker can fix (like C1). In 2 the cross-encoder did not promote it far enough (like C2). The rescued cases fail in four recognisable ways:
1. **Keyword and morphology traps (Q1, Q2, Q4).** The pooled embedding latches onto the most salient content word. "Start **lights**" pulls in every clause about lights. "**Rotary**" lands near "**rotating** parts". In Q4, "driver … during the race" matches the *Reverse gear* clause ("driven in reverse by the driver at any time during the Competition"). The cross-encoder scores these distractors at logit −5 to −11 because, reading query and clause together, it sees they don't answer the question.
2. **Prohibitions phrased abstractly (Q1, Q5, Q6).** The regulation never says "launch control" or "cheat". It says "any system … the purpose and/or effect of which is to detect when a race start signal is given" and "circumventing the tests". The cross-encoder links the user's intent to the abstract wording; the bi-encoder's single vector does not.
3. **Exceptions buried in a general clause (Q3).** The answer is the phrase "*except for* safety features" inside C1.5, a long paragraph about compliance duties. Three other chunks from the same article and the C3.3 legality-checking section look just as "on-topic" to the bi-encoder.
4. **Crowded sections (Q7).** C5.2 *Power Unit Energy Flow* is split into many sibling chunks that all talk about MJ limits. The bi-encoder cannot tell "state-of-charge window" apart from "recharge per lap"; the cross-encoder can, though only by a modest margin (#4 → #3).

**Answer 1 vs. Answer 2.** The same LLM was given the same prompt; only the context differed.

| Query | Answer 1 (baseline) | Answer 2 (re-ranked) | Effect of the missing clause |
|---|---|---|---|
| Q1 | "The excerpts do not contain enough information…" | "**No.** Any system whose purpose is to detect when a race start signal is given is not permitted [C8.3.1]" | **Omission → refusal** |
| Q2 | "Not enough information… none address engine architecture" | "**No.** Only 4-stroke engines with reciprocating pistons are permitted [C5.1.1]" | **Omission → refusal** |
| Q3 | "No", justified by the *electronic-systems* and CAD/measurement provisions | "No", and quotes the actual exception "**except for safety features**" [C1.5], although the explanation then wanders | **Omitted clause.** Right verdict, wrong legal basis: the decisive exception never reached the baseline |
| Q4 | "Not enough information… does not mention driver default cancellation" | "**No.** Any driver default turned on during the start lockout period may not be turned off before the end of that period [C8.2.4]" | **Omission → refusal** |
| Q5 | "Not enough information… no excerpt addresses intentional circumvention" | "**No.** Any system that achieves, seeks to, or is suspected of circumventing the tests is prohibited [C3.18.1]" | **Omission → refusal** |
| Q6 | "Not enough information…" | "**No.** The wheel fastener may not attach or mount any part except the wheel assembly [C10.9.1]" | **Omission → refusal** |
| Q7 | "The excerpts do not specify a permitted… swing", then lists the *wrong* limits (8.5 MJ recharge) | "The difference between max and min state of charge may not exceed **4 MJ** [C5.2.9]" | **Omission → near-miss.** The baseline offers related but wrong numbers |
| C1 | "Not enough information" | "**Yes**", inferred from a clutch-dependent rev limit and the clutch torque map (C5.14, C9.2.5) | **Re-ranking made it worse.** The true rule (C5.13.3, power unit control *must not* be influenced by clutch position) was never retrieved, and the cross-encoder promoted *related-but-misleading* clauses that invited a wrong inference |
| C2 | "Not enough information" | "Not enough information" | No change: C4.4 stayed at #6 |

**Takeaways.**
1. **Re-ranking fixed every case where the answer was in the top 10.** The baseline produced a usable answer for 0 of 7 failure queries (6 refusals, 1 right-for-the-wrong-reason). The re-ranked pipeline answered 7 of 7 correctly and cited the governing clause in all 7.
2. **This document fails differently from a short rulebook.** With a strict "answer only from the excerpts" prompt and a 1,000-chunk regulation, the baseline mostly fails *safely*, by refusing. That is still a real failure: the user is told the regulations are silent on something they explicitly prohibit. When related numbers *are* present (Q7), the baseline drifts towards quoting the wrong limit.
3. **Re-ranking is bounded by recall and can mislead (C1, C2).** A re-ranker can only re-order what stage 1 retrieved. In C1, the best of a bad candidate set was misleading, and the grounded LLM turned it into a confident wrong "Yes". In C2, the vocabulary gap ("top up oil or water" vs. "no substance may be added") was too big even for the cross-encoder. Stage-1 recall (a larger k, hybrid BM25 + dense search, query rewriting) still matters.
4. **Margins are sometimes thin.** Q4's gold clause beat the runner-up by only 0.1 logits, and Q6 and Q7 were promoted only to #3. With `TOP_N = 3` these still reach the LLM, but they show why practitioners often pass 4–5 re-ranked chunks.

*Note: LLM outputs come from the saved run (temperature 0, thinking disabled). Re-running may change the wording slightly, but the rankings are deterministic.*

### 10.3 Latency profiling

Measured on CPU with 4 threads, the 1,032-chunk corpus and 20 repeats per query (full table in §8):

| Stage | Pipeline 1 (baseline) | Pipeline 2 (re-rank) |
|---|---|---|
| Query embedding (bi-encoder) | 6.6 ms | 6.6 ms |
| FAISS exact search (1,032 × 384-d) | 0.1 ms | 0.1 ms |
| Cross-encoder, 10 pairs | – | 31.6 ms |
| **Retrieval total** | **≈ 7 ms** | **≈ 38 ms** (≈ 5.7× slower) |
| LLM generation (DeepSeek API, network) | ≈ 1.4 s | ≈ 1.2 s |
| **End-to-end** | **≈ 1.29 s** | **≈ 1.32 s** (+2.4%) |

Offline indexing (embedding all 1,032 chunks once) took about 2.1 s.

- Relative to retrieval alone, re-ranking is expensive: 10 full cross-attention forward passes (≈ 3.2 ms per pair) against a single query embedding plus a 0.1 ms exact vector search.
- Relative to the whole request it is cheap: the **+31 ms** is smaller than the **±170 ms** jitter of the LLM call. For a ~2% latency increase, the system went from 0/7 to 7/7 correctly grounded answers on the stress-test set.
- Exact FAISS search is still sub-millisecond at 1k vectors. At millions of vectors an ANN index (HNSW/IVF) keeps it in the low milliseconds, while the cross-encoder cost stays fixed at k = 10 pairs.
- The P1 vs. P2 difference in LLM latency is network noise, not a real effect: both prompts have identical structure and similar length.

### 10.4 Scaling question: why not run the cross-encoder over the entire collection?

The two architectures scale very differently, because of **when** the expensive work happens.

- **Bi-encoder: pre-computation plus sub-linear search.** Chunk embeddings do not depend on the query, so all N chunks are embedded **once, offline** (≈ 2.1 s for this 254-page regulation) and stored in an index. At query time the cost is **one** encoder pass for the query plus a nearest-neighbour search, which ANN structures (HNSW, IVF) make roughly **O(log N)**. The cost per query barely grows with the corpus.
- **Cross-encoder: no pre-computation is possible.** The relevance score comes from self-attention over the *concatenated* query and chunk, so it cannot be split into a query part and a document part and cached. Every new query needs **N full transformer forward passes**, each over ~150–300 tokens with attention that is quadratic in sequence length. The cost per query is **O(N)**, with a large constant.

**Measured, not just argued.** §9 cross-encoded *every* chunk of this regulation for one query. That took **2.4 s (≈ 2.35 ms per pair)**, versus 6.6 ms for bi-encoder retrieval and 38 ms for the two-stage pipeline. Extrapolating the measured per-pair cost:

| Corpus size | Cross-encode everything (per query) | Two-stage: retrieve top 10 → re-rank 10 |
|---|---|---|
| 1,032 chunks (this regulation) | 2.4 s | ≈ 38 ms |
| 10,000 | 23.5 s | ≈ 38 ms + ANN |
| 100,000 | ~4 min | ≈ 38 ms + ANN |
| 10,000,000 (web-scale) | ~6.5 h | ≈ 38 ms + ANN |

Even for this *single* document, a full cross-encoder scan is **60× slower** than two-stage retrieval. A GPU would help by perhaps 10–100×, but the cost would still grow linearly with every document added, and compute would be burned on thousands of obviously irrelevant clauses (brake callipers when the question is about fuel).

**The two-stage "retrieve then re-rank" design gets the best of both.**
- The cheap, high-**recall** bi-encoder narrows N chunks down to a handful of candidates.
- The expensive, high-**precision** cross-encoder spends its budget only on that shortlist, where its fine-grained reading of negations, exceptions and abstract prohibitions matters (Q1–Q7).
- The re-ranking cost is constant in the corpus size: it depends only on k.

The trade-off, shown by control C1, is that the re-ranker's quality is capped by stage-1 recall. In practice k is tuned (often 20–100) to balance recall against re-ranking latency.